# Double Descent and the Interpolation Threshold

Sweep capacity and noise using actual fitted predictors and untouched test data.

## Before You Start

Course 11. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** dynamics-04_probes.

## What You Will Be Able to Do

- Fit minimum-norm and ridge predictors
- Measure model-wise interpolation
- Distinguish double descent from grokking

## Capacity changes the regime

Classical bias-variance intuition predicts an intermediate complexity sweet spot. Some overparameterized systems show a second improvement after a peak near the capacity needed to interpolate training data. This is model-wise double descent when capacity varies. Epoch-wise double descent varies training time instead. Neither curve is universal; noise, features, regularization, optimizer, and sample size matter.

For design matrix X [N,D] and targets y [N,K], the minimum-norm least-squares solution uses a pseudoinverse or an SVD-based least-squares solver. When D>N, many weight vectors may fit the training data; minimum norm chooses one. Ridge minimizes ||XW-y|| squared + lambda||W|| squared. Our implementation uses an augmented least-squares system, avoiding an explicit matrix inverse and working on both sides of the interpolation threshold.

For repeated training sets, average their fitted predictions at each fixed test point. Bias squared is the squared difference of that mean from noiseless truth. Variance is the mean squared deviation around that mean prediction. Averaged over test points, MSE equals bias squared plus variance; independent observation noise adds its own variance. Polynomial features [1,x,x^2,...,x^degree] let the experiment vary capacity, but high-degree monomial bases can be ill-conditioned.

## An honest sweep

Generate one teacher and disjoint train/test inputs. Add noise only according to a declared recipe. Sweep D around N using nested random features, fit on train, and report both train and test MSE over several seeds. Random-label experiments remove meaningful signal; near-zero training loss then indicates memorization, not useful learning. More data moves the interpolation threshold and can change the entire curve.

Grokking is delayed generalization along a training trajectory. Double descent is a nonmonotonic error pattern over capacity or time. They can share ingredients such as regularization but are not synonyms. A smooth monotonic curve in your small experiment is a valid result. Do not fabricate a peak or infer a universal scaling law from a single trial.

## Worked Example: Interpolation is not identification

One training row X=[1,1], target y=2.

1. **Many fits:** Any weights with w1+w2=2 fit perfectly, including [2,0] and [100,-98].
2. **Minimum norm:** [1,1] has the smallest squared norm among these fits.
3. **Held-out behavior:** For test x=[1,-1], these fits give 2,198, and 0 respectively. Equal training error does not imply equal generalization.

The solution-selection rule matters after the training constraints become underdetermined.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Least-squares capacity studies with a stable solver."""
import numpy as np

def ridge_fit(x, y, penalty=0.0):
    """Fit W [D,K] from X [N,D], Y [N,K], without an intercept.

Objective ||XW-Y||^2 + penalty*||W||^2. Nonnegative finite penalty required.
Assume nonempty finite arrays with compatible shapes. At zero penalty use
the minimum-norm solution, including rank-deficient and D>N designs."""
    raise NotImplementedError('Implement this function, then run the checks.')

def bias_variance(predictions, truth):
    """Return bias2, variance, mse for predictions [runs,points] and truth [points].

Fixed noiseless truth; population variance over runs. MSE averages all runs
and points. Thus mse = bias2 + variance (no observation-noise term here).
Valid finite nonempty shapes assumed."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

X=[1,1], y=2, minimum Euclidean norm.

- First fitted weight

<details><summary>Worked answers</summary>

- 1: Symmetry and the sum constraint give [1,1].

</details>

## Experiment

Run `experiments/dynamics_interpolation.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Print per-seed train/test MSE for capacities below, near, and above N, with and without ridge.
- The finite random-feature experiment is not a replication of deep-network benchmark curves.

## Transfer Problems

1. Repeat after doubling the number of training examples.
2. Increase label noise and examine the peak and the effect of regularization.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[Deep Double Descent](https://arxiv.org/abs/1912.02292). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Least-squares capacity studies with a stable solver."""
import numpy as np

def ridge_fit(x, y, penalty=0.0):
    """Fit W [D,K] from X [N,D], Y [N,K], without an intercept.

Objective ||XW-Y||^2 + penalty*||W||^2. Nonnegative finite penalty required.
Assume nonempty finite arrays with compatible shapes. At zero penalty use
the minimum-norm solution, including rank-deficient and D>N designs."""
    raise NotImplementedError('Implement this function, then run the checks.')

def bias_variance(predictions, truth):
    """Return bias2, variance, mse for predictions [runs,points] and truth [points].

Fixed noiseless truth; population variance over runs. MSE averages all runs
and points. Thus mse = bias2 + variance (no observation-noise term here).
Valid finite nonempty shapes assumed."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
assert np.allclose(ridge_fit(np.array([[1., 1.]]), np.array([[2.]])), [[1], [1]])
rng = np.random.default_rng(9)
for n, d in [(9, 4), (4, 9), (5, 5)]:
    x, y = rng.normal(size=(n, d)), rng.normal(size=(n, 2))
    for penalty in [0., .3, 4.]:
        w = ridge_fit(x, y, penalty)
        assert np.allclose((x.T@x+penalty*np.eye(d))@w, x.T@y, atol=1e-7)
result = bias_variance(np.array([[1., 4.], [3., 2.]]), np.array([1., 2.]))
assert result == dict(bias2=1., variance=1., mse=2.)
print('Interpolation checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Least-squares capacity studies with a stable solver."""
import numpy as np


def ridge_fit(x, y, penalty=0.):
    """Fit W [D,K] from X [N,D], Y [N,K], without an intercept.

    Objective ||XW-Y||^2 + penalty*||W||^2. Nonnegative finite penalty required.
    Assume nonempty finite arrays with compatible shapes. At zero penalty use
    the minimum-norm solution, including rank-deficient and D>N designs.
    """
    if not np.isfinite(penalty) or penalty < 0:
        raise ValueError('invalid penalty')
    d = x.shape[1]
    augmented = np.vstack([x, np.sqrt(penalty)*np.eye(d)])
    targets = np.vstack([y, np.zeros((d, y.shape[1]))])
    return np.linalg.lstsq(augmented, targets, rcond=None)[0]


def bias_variance(predictions, truth):
    """Return bias2, variance, mse for predictions [runs,points] and truth [points].

    Fixed noiseless truth; population variance over runs. MSE averages all runs
    and points. Thus mse = bias2 + variance (no observation-noise term here).
    Valid finite nonempty shapes assumed.
    """
    mean = predictions.mean(axis=0)
    return dict(bias2=float(((mean-truth)**2).mean()), variance=float(((predictions-mean)**2).mean()), mse=float(((predictions-truth)**2).mean()))


In [ ]:
import numpy as np
assert np.allclose(ridge_fit(np.array([[1., 1.]]), np.array([[2.]])), [[1], [1]])
rng = np.random.default_rng(9)
for n, d in [(9, 4), (4, 9), (5, 5)]:
    x, y = rng.normal(size=(n, d)), rng.normal(size=(n, 2))
    for penalty in [0., .3, 4.]:
        w = ridge_fit(x, y, penalty)
        assert np.allclose((x.T@x+penalty*np.eye(d))@w, x.T@y, atol=1e-7)
result = bias_variance(np.array([[1., 4.], [3., 2.]]), np.array([1., 2.]))
assert result == dict(bias2=1., variance=1., mse=2.)
print('Interpolation checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Random-feature capacity sweeps plus repeated-data polynomial bias/variance."""
import json
import numpy as np


def interpolation_experiment():
    results = []
    n = 24
    for seed in [1, 7, 19]:
        rng = np.random.default_rng(seed)
        train_x, test_x = rng.normal(size=(n, 5)), rng.normal(size=(256, 5))
        teacher = rng.normal(size=5)
        clean, test_y = train_x@teacher, test_x@teacher
        bank = rng.normal(size=(5, 96))
        for noise in [0., .5]:
            y = clean + rng.normal(size=n)*noise
            for d in [8, 16, 23, 24, 25, 32, 48, 96]:
                x = np.maximum(train_x@bank[:, :d], 0) / np.sqrt(d)
                xt = np.maximum(test_x@bank[:, :d], 0) / np.sqrt(d)
                for penalty in [0., .1]:
                    w = np.linalg.lstsq(np.vstack([x, np.sqrt(penalty)*np.eye(d)]), np.concatenate([y, np.zeros(d)]), rcond=None)[0]
                    results.append(dict(seed=seed, n=n, width=d, noise=noise, ridge=penalty, train_mse=float(((x@w-y)**2).mean()), test_mse=float(((xt@w-test_y)**2).mean())))
    # Repeated noisy training sets, fixed noiseless target: exact empirical decomposition.
    rng = np.random.default_rng(94)
    grid = np.linspace(-1, 1, 80); truth = np.sin(3*grid)
    polynomial = []
    for degree in [2, 6, 12, 24]:
        predictions = []
        for _ in range(20):
            x = rng.uniform(-1, 1, 16); y = np.sin(3*x)+rng.normal(size=16)*.15
            w = np.linalg.lstsq(np.vander(x, degree+1, increasing=True), y, rcond=None)[0]
            predictions.append(np.vander(grid, degree+1, increasing=True)@w)
        predictions = np.array(predictions); mean = predictions.mean(0)
        bias2 = float(((mean-truth)**2).mean()); variance = float(((predictions-mean)**2).mean())
        mse = float(((predictions-truth)**2).mean())
        assert np.isclose(mse, bias2+variance)
        polynomial.append(dict(degree=degree, bias2=bias2, variance=variance, mse=mse))
    random_design=rng.normal(size=(16,32));random_labels=rng.normal(size=(16,1))
    random_weights=np.linalg.lstsq(random_design,random_labels,rcond=None)[0]
    random_test=rng.normal(size=(128,32));random_test_labels=rng.normal(size=(128,1))
    random_control=dict(train_mse=float(((random_design@random_weights-random_labels)**2).mean()),test_mse=float(((random_test@random_weights-random_test_labels)**2).mean()))
    assert random_control['train_mse']<1e-20
    return dict(capacity_sweep=results, polynomial_bias_variance=polynomial,random_label_control=random_control,
        limitation='No universal second descent is asserted. Monomial bases become ill-conditioned; solver and regularization matter.')


if __name__ == '__main__':
    print(json.dumps(interpolation_experiment(), indent=2))
